In [ ]:
# For turning off unused rxns, so they stay in the GSM model but aren't used in RBA. Optional, but make sure you know all potential conditions you'll be testing.
## included separately from model reduction in case you've updated the GSM model and want to still preserve its changes somewhere else w/o 
import cobra,json,os,sys,re,pandas as pd, io, requests
from collections import OrderedDict
from copy import deepcopy

# keep going up until the "pycore" folder is found
wd = os.getcwd()
os.chdir('..')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *

os.chdir(wd)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [2]:
# change path below to whichever version you'd like to reference (e.g., modified version for max yield calculations)
model_reference = load_cobra_model('../application/input/iIsor850_RBA_pathwayadd_reduced.json')
# save backup of model before turning off rxns

for r in model.reactions:
	for note in ('turned_off_for_RBA','altered_for_RBA'):
		if getattr(r, 'notes', {}).get(note, False):
			# set lower and upper bounds to "Original bounds were (x,y)"
			original_bounds = re.search(r'Original bounds were \(([^,]*),([^)]*)\)', r.notes.get(note,''))
			if original_bounds:
				r.lower_bound = float(original_bounds.group(1))
				r.upper_bound = float(original_bounds.group(2))
			# remove the note
			del r.notes[note]
			# if notes is now empty, remove the notes attribute
			# if not r.notes:
			# 	del r.notes

save_cobra_model(model, '_full'.join(os.path.splitext(SM_path)))
# turn off rxns in model that aren't found in model_reference; clarify in notes section why this was done (and the original bounds)
for rxn in model.reactions:
	if rxn.bounds == (0,0):
		continue
	if rxn.id in model_reference.reactions:
		if model_reference.reactions.get_by_id(rxn.id).bounds == (0,0):
			rxn.notes['altered_for_RBA'] = f'Turned off for Resource Balance Analysis (RBA) model building, because it was found to be turned off in the reference model ({model_reference.name}) used for max yield calculations. Original bounds were {rxn.bounds}.'
			print(f'Turned off reaction {rxn.id} because it was turned off in the reference model ({model_reference.name}) used for max yield calculations. Original bounds were {rxn.bounds}.')
			rxn.bounds = (0,0)
		elif model_reference.reactions.get_by_id(rxn.id).bounds != rxn.bounds:
			rxn.notes['altered_for_RBA'] = f'Bounds were changed for Resource Balance Analysis (RBA) model building, because they were found to be different in the reference model ({model_reference.name}) used for max yield calculations. Original bounds were {rxn.bounds}, new bounds are {model_reference.reactions.get_by_id(rxn.id).bounds}.'
			print(f'Changed bounds for reaction {rxn.id} because they were different in the reference model ({model_reference.name}) used for max yield calculations. Original bounds were {rxn.bounds}, new bounds are {model_reference.reactions.get_by_id(rxn.id).bounds}.')
			rxn.bounds = model_reference.reactions.get_by_id(rxn.id).bounds
		continue
	else:
		rxn.notes['altered_for_RBA'] = f'Turned off for Resource Balance Analysis (RBA) model building, because it was not found in the reference model ({model_reference.name}) used for max yield calculations. Original bounds were {rxn.bounds}.'
		print(f'Turned off reaction {rxn.id} because it was not found in the reference model ({model_reference.name}) used for max yield calculations. Original bounds were {rxn.bounds}.')
		rxn.bounds = (0,0)
save_cobra_model(model, SM_path)

Changed bounds for reaction ARABDt_c because they were different in the reference model (iIsor850) used for max yield calculations. Original bounds were (-1000.0, 1000.0), new bounds are (0.0, 1000.0).
Turned off reaction EX_arab__D_e because it was turned off in the reference model (iIsor850) used for max yield calculations. Original bounds were (0.0, 1000.0).
Turned off reaction EX_csn_e because it was turned off in the reference model (iIsor850) used for max yield calculations. Original bounds were (0.0, 1000.0).
Changed bounds for reaction CYTK2_c because they were different in the reference model (iIsor850) used for max yield calculations. Original bounds were (-1000.0, 1000.0), new bounds are (-1000.0, 0.0).
Turned off reaction EX_cytd_e because it was turned off in the reference model (iIsor850) used for max yield calculations. Original bounds were (0.0, 1000.0).
Turned off reaction EX_fru_e because it was turned off in the reference model (iIsor850) used for max yield calculati